<a href="https://colab.research.google.com/github/weagan/ARC-AGI-3/blob/main/duck_harness_prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Duck Harness Prototype for ARC-AGI-3
This notebook provides a high-level executable prototype of the **Duck Harness** architecture designed by Tufa Labs for the ARC-AGI-3 environment. It showcases:
1. Multi-platform secret handling (Google Colab vs Kaggle).
2. Dual-modal 2D grid visualization/serialization.
3. A code-as-action Python REPL execution loop.
4. Automatic context window eviction supervision.

### 1. Platform-Agnostic Secret Handling
Detects the current environment (Colab or Kaggle) and loads the required API keys securely without hardcoding.

In [ ]:
import os

def load_api_key(secret_name="HF_TOKEN"):
    """Loads a secret securely from Google Colab or Kaggle secret managers."""
    # Check for Google Colab
    try:
        from google.colab import userdata
        print("[Info] Google Colab environment detected. Fetching secret...")
        return userdata.get(secret_name)
    except ImportError:
        pass

    # Check for Kaggle
    try:
        from kaggle_secrets import UserSecretsClient
        print("[Info] Kaggle environment detected. Fetching secret...")
        user_secrets = UserSecretsClient()
        return user_secrets.get_secret(secret_name)
    except ImportError:
        pass

    # Fallback to local environment variables
    print("[Info] Local or fallback environment detected.")
    return os.environ.get(secret_name, "MOCK_API_KEY_FOR_LOCAL_RUNNING")

API_KEY = load_api_key()
print("Secret status: Loaded successfully (masked for privacy).")

[Info] Google Colab environment detected. Fetching secret...
Secret status: Loaded successfully (masked for privacy).


### 2. The Mock Game Environment
A lightweight environment representing an ARC-AGI style grid game where the agent must infer rules and execute optimal spatial choices.

In [ ]:
import numpy as np

class ArcGameEnvironment:
    """Simulates a simple 2D ARC-style maze where the goal must be discovered."""
    def __init__(self):
        # 0 = Empty, 1 = Wall, 2 = Player, 3 = Goal
        self.grid = np.array([
            [1, 1, 1, 1, 1],
            [1, 2, 0, 0, 1],
            [1, 1, 1, 0, 1],
            [1, 3, 0, 0, 1],
            [1, 1, 1, 1, 1]
        ])
        self.player_pos = (1, 1)
        self.moves = 0

    def get_state(self):
        return self.grid.copy()

    def step(self, direction):
        """Executes a step: 'UP', 'DOWN', 'LEFT', 'RIGHT'"""
        self.moves += 1
        r, c = self.player_pos
        if direction == 'UP': r -= 1
        elif direction == 'DOWN': r += 1
        elif direction == 'LEFT': c -= 1
        elif direction == 'RIGHT': c += 1

        # Check constraints / collision
        if self.grid[r, c] != 1:
            # Move current position
            self.grid[self.player_pos] = 0
            if self.grid[r, c] == 3:
                print("Goal reached autonomously!")
            self.player_pos = (r, c)
            self.grid[r, c] = 2

        return self.get_state(), self.moves

### 3. Duck Harness Supervisor Architecture
Implements the Python REPL interface, text serialization, and history eviction supervision to prevent context explosion.

In [ ]:
import sys
from io import StringIO

class DuckHarness:
    def __init__(self, max_context_steps=3):
        self.max_context_steps = max_context_steps
        self.history = []

    def serialize_grid(self, grid):
        """Converts the numerical 2D array into a structural textual string format."""
        return "\n".join([" ".join(map(str, row)) for row in grid])

    def supervise_context(self):
        """Evicts stale historical execution contexts to prevent attention decay."""
        if len(self.history) > self.max_context_steps:
            print(f"[Harness] Evicting oldest step context. Current size: {len(self.history)}")
            self.history = self.history[-self.max_context_steps:]

    def execute_code_repl(self, code_str, env):
        """Runs code generated by the LLM within a safe local interpreter state."""
        old_stdout = sys.stdout
        redirected_output = StringIO()
        sys.stdout = redirected_output

        local_vars = {"env": env}
        try:
            exec(code_str, {}, local_vars)
            error = None
        except Exception as e:
            error = str(e)

        sys.stdout = old_stdout
        return redirected_output.getvalue(), error

### 4. Running the Code-as-Action Loop Simulation
Simulates the model generating a Python pathfinding script step to interact with the environment through the harness layer.

In [ ]:
# Instantiate environment and harness supervisor
env = ArcGameEnvironment()
harness = DuckHarness(max_context_steps=2)

# Initial state capture
current_grid = env.get_state()
serialized_view = harness.serialize_grid(current_grid)
harness.history.append({"state": serialized_view})

print("--- Initial Game View ---")
print(serialized_view)

# Simulated programmatic action sequence output by the model reasoning layer
llm_generated_code = """
# Model inferred rule: 2 must reach 3 avoiding 1
env.step('RIGHT')
env.step('RIGHT')
env.step('DOWN')
env.step('DOWN')
env.step('LEFT')
"""

print("\n--- Executing LLM Code-as-Action via Harness REPL ---")
output, err = harness.execute_code_repl(llm_generated_code, env)

if err:
    print(f"Execution Error: {err}")
else:
    print(output if output else "Code executed smoothly with no output noise.")

final_grid = env.get_state()
print("\n--- Final Game View ---")
print(harness.serialize_grid(final_grid))
harness.history.append({"state": harness.serialize_grid(final_grid)})
harness.supervise_context()

--- Initial Game View ---
1 1 1 1 1
1 2 0 0 1
1 1 1 0 1
1 3 0 0 1
1 1 1 1 1

--- Executing LLM Code-as-Action via Harness REPL ---
Code executed smoothly with no output noise.

--- Final Game View ---
1 1 1 1 1
1 0 0 0 1
1 1 1 0 1
1 3 2 0 1
1 1 1 1 1
